In [ ]:
# ============================================================
# STEP 1 — Caricamento, pulizia e preparazione EU-LS 2022
# Does Social Media Use Reduce Loneliness?
# ============================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

os.makedirs('output/dataset', exist_ok=True)
os.makedirs('output/figures', exist_ok=True)

# ------------------------------------------------------------------
# 1.1  CARICAMENTO
# ------------------------------------------------------------------
df_raw = pd.read_csv('eu_loneliness_survey_eu27_values.csv', low_memory=False)
print(f"Dataset grezzo: {df_raw.shape[0]} righe × {df_raw.shape[1]} colonne")

# ------------------------------------------------------------------
# 1.2  SELEZIONE COLONNE DI INTERESSE
# ------------------------------------------------------------------
COLS = ['loneliness_ucla_a', 'loneliness_ucla_b', 'loneliness_ucla_c',
        'sm_time_a', 'country', 'age', 'gender', 'education', 'income_decile']

df = df_raw[COLS].copy()

# ------------------------------------------------------------------
# 1.3  COSTRUZIONE score_UCLA (variabile dipendente)
# ------------------------------------------------------------------
# Scala UCLA 3 item, valori 1–3 (999 = non risposta)
# Score = somma dei 3 item → range 3–9 (9 = massima solitudine)

for col in ['loneliness_ucla_a', 'loneliness_ucla_b', 'loneliness_ucla_c']:
    df[col] = df[col].replace(999, np.nan)

df['score_UCLA'] = df[['loneliness_ucla_a',
                        'loneliness_ucla_b',
                        'loneliness_ucla_c']].sum(axis=1, min_count=3)

print("\n--- score_UCLA ---")
print(df['score_UCLA'].describe().round(2))
print(f"Missing: {df['score_UCLA'].isna().sum()} ({df['score_UCLA'].isna().mean()*100:.1f}%)")

# ------------------------------------------------------------------
# 1.4  COSTRUZIONE intensita_sm (variabile indipendente — ORDINALE)
# ------------------------------------------------------------------
# sm_time_a: 1–8 = livelli crescenti di tempo sui social media
#            999  = non usa social media → ricodificato come 0
# Variabile finale: 0 (non utente) + 1–8 (utente per intensità)
# Range finale: 0–8 (0 = mai, 8 = uso massimo)

df['intensita_sm'] = df['sm_time_a'].replace(999, 0)

print("\n--- intensita_sm ---")
print(df['intensita_sm'].value_counts().sort_index())
print(f"\nMedia: {df['intensita_sm'].mean():.2f}")
print(f"Missing: {df['intensita_sm'].isna().sum()}")

# ------------------------------------------------------------------
# 1.5  COSTRUZIONE fascia_eta (variabile moderatrice)
# ------------------------------------------------------------------
bins   = [15, 34, 54, 74, 120]
labels = ['16–34', '35–54', '55–74', '75+']

df['fascia_eta'] = pd.cut(df['age'], bins=bins, labels=labels, right=True)

print("\n--- Distribuzione fasce d'età ---")
print(df['fascia_eta'].value_counts().sort_index())

# ------------------------------------------------------------------
# 1.6  PULIZIA VARIABILI DI CONTROLLO
# ------------------------------------------------------------------
for col in ['gender', 'education', 'income_decile']:
    df[col] = df[col].replace(999, np.nan)

print("\n--- Missing variabili di controllo ---")
for col in ['gender', 'education', 'income_decile']:
    n = df[col].isna().sum()
    print(f"  {col:20s}: {n} ({n/len(df)*100:.1f}%)")

# ------------------------------------------------------------------
# 1.7  LISTWISE DELETION
# ------------------------------------------------------------------
COLONNE_MODELLO = ['score_UCLA', 'intensita_sm', 'age', 'fascia_eta',
                   'gender', 'education', 'income_decile', 'country']

n_prima = len(df)
df_clean = df.dropna(subset=COLONNE_MODELLO).copy()
n_dopo   = len(df_clean)

print(f"\n--- Listwise deletion ---")
print(f"Righe prima : {n_prima}")
print(f"Righe dopo  : {n_dopo}")
print(f"Rimosse     : {n_prima - n_dopo} ({(n_prima - n_dopo)/n_prima*100:.1f}%)")

# ------------------------------------------------------------------
# 1.8  DATAFRAME FINALE
# ------------------------------------------------------------------
COLONNE_FINALI = ['country', 'intensita_sm', 'score_UCLA',
                  'fascia_eta', 'age', 'gender', 'education', 'income_decile']

df_final = df_clean[COLONNE_FINALI].reset_index(drop=True)
df_final.columns = ['paese', 'intensita_sm', 'score_UCLA',
                    'fascia_eta', 'eta', 'sesso', 'education', 'income']

print("\n=== DataFrame finale ===")
print(f"Shape: {df_final.shape}")
print(f"Paesi presenti: {df_final['paese'].nunique()}")
print(f"\nAnteprima:\n{df_final.head(5)}")

# ------------------------------------------------------------------
# 1.9  GRAFICI DIAGNOSTICI
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribuzione score UCLA
axes[0].hist(df_final['score_UCLA'], bins=7, color='steelblue', edgecolor='white')
axes[0].set_title('Distribuzione score UCLA')
axes[0].set_xlabel('Score (3–9)')
axes[0].set_ylabel('Frequenza')

# Distribuzione intensita_sm (ordinale 0–8)
conteggi = df_final['intensita_sm'].value_counts().sort_index()
axes[1].bar(conteggi.index, conteggi.values, color='steelblue', edgecolor='white')
axes[1].set_title('Distribuzione intensità uso social media')
axes[1].set_xlabel('Livello (0 = non utente, 8 = uso massimo)')
axes[1].set_ylabel('Frequenza')
axes[1].set_xticks(range(9))

# Score UCLA medio per livello di intensita_sm
medie_globali = df_final.groupby('intensita_sm')['score_UCLA'].mean()
axes[2].plot(medie_globali.index, medie_globali.values,
             marker='o', color='steelblue', linewidth=2)
axes[2].set_title('Score UCLA medio per intensità social media')
axes[2].set_xlabel('Livello intensita_sm')
axes[2].set_ylabel('Score UCLA medio')
axes[2].set_xticks(range(9))

plt.tight_layout()
fig.savefig('output/figures/step1_diagnostici.png', dpi=150, bbox_inches='tight')
plt.show()

# ------------------------------------------------------------------
# 1.10  SALVATAGGIO
# ------------------------------------------------------------------
df_final.to_csv('output/dataset/eu_ls_clean.csv', index=False)
print("\nSalvato: output/dataset/eu_ls_clean.csv")
print("Salvato: output/figures/step1_diagnostici.png")

In [ ]:
# ============================================================
# STEP 2 — Analisi esplorativa (EDA)
# Does Social Media Use Reduce Loneliness?
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.float_format', '{:.2f}'.format)

# ------------------------------------------------------------------
# 2.1  CARICAMENTO
# ------------------------------------------------------------------
df = pd.read_csv('output/dataset/eu_ls_clean.csv')
df['fascia_eta'] = pd.Categorical(df['fascia_eta'],
                                   categories=['16–34', '35–54', '55–74', '75+'],
                                   ordered=True)
print(f"Dataset: {df.shape[0]} righe × {df.shape[1]} colonne")

# ------------------------------------------------------------------
# 2.2  STATISTICHE DESCRITTIVE PER FASCIA D'ETÀ
# ------------------------------------------------------------------
desc = df.groupby('fascia_eta', observed=True).agg(
    n               = ('score_UCLA', 'count'),
    ucla_media      = ('score_UCLA', 'mean'),
    ucla_std        = ('score_UCLA', 'std'),
    sm_media        = ('intensita_sm', 'mean'),
    sm_std          = ('intensita_sm', 'std'),
).round(2)

print("\n=== Statistiche per fascia d'età ===")
print(desc)

# ------------------------------------------------------------------
# 2.3  STATISTICHE DESCRITTIVE PER PAESE
# ------------------------------------------------------------------
desc_paese = df.groupby('paese', observed=True).agg(
    n           = ('score_UCLA', 'count'),
    ucla_media  = ('score_UCLA', 'mean'),
    sm_media    = ('intensita_sm', 'mean'),
).round(2).sort_values('ucla_media', ascending=False)

print("\n=== Statistiche per paese (ordinato per UCLA medio) ===")
print(desc_paese)

# ------------------------------------------------------------------
# 2.4  CORRELAZIONE BIVARIATA — tutto il campione
# ------------------------------------------------------------------
pearson_r, pearson_p   = stats.pearsonr(df['intensita_sm'], df['score_UCLA'])
spearman_r, spearman_p = stats.spearmanr(df['intensita_sm'], df['score_UCLA'])

print("\n=== Correlazione bivariata (tutto il campione) ===")
print(f"Pearson  r = {pearson_r:.3f}  p = {pearson_p:.4f}")
print(f"Spearman r = {spearman_r:.3f}  p = {spearman_p:.4f}")

# ------------------------------------------------------------------
# 2.5  CORRELAZIONE PER FASCIA D'ETÀ
# ------------------------------------------------------------------
print("\n=== Correlazione Pearson per fascia d'età ===")
corr_per_fascia = []
for fascia in ['16–34', '35–54', '55–74', '75+']:
    sub = df[df['fascia_eta'] == fascia]
    r, p = stats.pearsonr(sub['intensita_sm'], sub['score_UCLA'])
    corr_per_fascia.append({'fascia': fascia, 'n': len(sub),
                            'pearson_r': round(r, 3), 'p_value': round(p, 4)})
    print(f"  {fascia:6s}  r = {r:.3f}  p = {p:.4f}  (n={len(sub)})")

# ------------------------------------------------------------------
# 2.6  GRAFICI EDA
# ------------------------------------------------------------------
fig = plt.figure(figsize=(16, 12))

# --- Fig A: Heatmap UCLA medio per paese × fascia d'età ---
ax1 = fig.add_subplot(2, 2, 1)
pivot = df.groupby(['paese', 'fascia_eta'],
                    observed=True)['score_UCLA'].mean().unstack()
sns.heatmap(pivot, ax=ax1, cmap='YlOrRd', annot=True, fmt='.1f',
            linewidths=0.3, cbar_kws={'label': 'Score UCLA medio'})
ax1.set_title('Score UCLA medio per paese × fascia d\'età')
ax1.set_xlabel('Fascia d\'età')
ax1.set_ylabel('Paese (codice numerico)')

# --- Fig B: Line plot UCLA medio per livello intensita_sm × fascia d'età ---
ax2 = fig.add_subplot(2, 2, 2)
colori = {'16–34': '#4C72B0', '35–54': '#DD8452',
          '55–74': '#55A868', '75+': '#C44E52'}
for fascia in ['16–34', '35–54', '55–74', '75+']:
    sub = df[df['fascia_eta'] == fascia]
    medie = sub.groupby('intensita_sm')['score_UCLA'].mean()
    ax2.plot(medie.index, medie.values, marker='o', linewidth=2,
             label=fascia, color=colori[fascia])
ax2.set_title('Score UCLA medio per intensità SM × fascia d\'età')
ax2.set_xlabel('Livello intensita_sm (0 = non utente, 8 = uso massimo)')
ax2.set_ylabel('Score UCLA medio')
ax2.set_xticks(range(9))
ax2.legend(title='Fascia d\'età')
ax2.axhline(df['score_UCLA'].mean(), color='gray',
            linestyle='--', linewidth=0.8, label='Media globale')

# --- Fig C: Boxplot score UCLA per fascia d'età ---
ax3 = fig.add_subplot(2, 2, 3)
df.boxplot(column='score_UCLA', by='fascia_eta', ax=ax3,
           boxprops=dict(color='steelblue'),
           medianprops=dict(color='red', linewidth=2),
           whiskerprops=dict(color='steelblue'),
           capprops=dict(color='steelblue'),
           flierprops=dict(marker='o', markersize=2, alpha=0.3))
ax3.set_title('Distribuzione score UCLA per fascia d\'età')
ax3.set_xlabel('Fascia d\'età')
ax3.set_ylabel('Score UCLA')
plt.suptitle('')

# --- Fig D: Boxplot score UCLA per livello intensita_sm ---
ax4 = fig.add_subplot(2, 2, 4)
df.boxplot(column='score_UCLA', by='intensita_sm', ax=ax4,
           boxprops=dict(color='steelblue'),
           medianprops=dict(color='red', linewidth=2),
           whiskerprops=dict(color='steelblue'),
           capprops=dict(color='steelblue'),
           flierprops=dict(marker='o', markersize=2, alpha=0.3))
ax4.set_title('Distribuzione score UCLA per livello intensita_sm')
ax4.set_xlabel('Livello intensita_sm')
ax4.set_ylabel('Score UCLA')
plt.suptitle('')

plt.tight_layout()
fig.savefig('output/figures/step2_eda.png', dpi=150, bbox_inches='tight')
plt.show()

# ------------------------------------------------------------------
# 2.7  RIEPILOGO RISULTATI EDA
# ------------------------------------------------------------------
print("\n=== Riepilogo EDA ===")
print(f"Score UCLA medio globale       : {df['score_UCLA'].mean():.2f}")
print(f"Intensita_sm media globale     : {df['intensita_sm'].mean():.2f}")
print(f"Correlazione Pearson globale   : r = {pearson_r:.3f} (p = {pearson_p:.4f})")
print(f"Correlazione Spearman globale  : r = {spearman_r:.3f} (p = {spearman_p:.4f})")
print("\nFascia con UCLA più alto       :", desc['ucla_media'].idxmax())
print("Fascia con UCLA più basso      :", desc['ucla_media'].idxmin())
print("Fascia con SM più intenso      :", desc['sm_media'].idxmax())
print("Fascia con SM meno intenso     :", desc['sm_media'].idxmin())

In [ ]:
# ============================================================
# STEP 3 — Regressione OLS individuale
# Does Social Media Use Reduce Loneliness?
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import statsmodels.formula.api as smf
from scipy import stats

pd.set_option('display.float_format', '{:.4f}'.format)

# ------------------------------------------------------------------
# 3.1  CARICAMENTO
# ------------------------------------------------------------------
df = pd.read_csv('output/dataset/eu_ls_clean.csv')
df['fascia_eta'] = pd.Categorical(df['fascia_eta'],
                                   categories=['16–34', '35–54', '55–74', '75+'],
                                   ordered=True)

# Termine quadratico
df['intensita_sm2'] = df['intensita_sm'] ** 2

# Dummies paese (paese 1 = riferimento, escluso automaticamente da C())
print(f"Dataset: {df.shape[0]} righe · {df['paese'].nunique()} paesi")

# ------------------------------------------------------------------
# 3.2  MODELLO OLS — TUTTO IL CAMPIONE
# ------------------------------------------------------------------
formula_full = ('score_UCLA ~ intensita_sm + intensita_sm2 '
                '+ sesso + education + income + C(paese)')

model_full  = smf.ols(formula_full, data=df).fit(cov_type='HC3')

print("\n=== OLS — tutto il campione ===")
print(f"N = {int(model_full.nobs)}  |  R² = {model_full.rsquared:.4f}"
      f"  |  R² adj = {model_full.rsquared_adj:.4f}")
print(f"\nCoefficienti chiave:")
chiavi = ['Intercept', 'intensita_sm', 'intensita_sm2',
          'sesso', 'education', 'income']
for k in chiavi:
    b  = model_full.params[k]
    se = model_full.bse[k]
    p  = model_full.pvalues[k]
    ci_lo, ci_hi = model_full.conf_int().loc[k]
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
    print(f"  {k:20s}  β={b:7.4f}  SE={se:.4f}  p={p:.4f}  "
          f"CI=[{ci_lo:.4f}, {ci_hi:.4f}]  {sig}")

# ------------------------------------------------------------------
# 3.3  MODELLO OLS — SEPARATO PER FASCIA D'ETÀ
# ------------------------------------------------------------------
fasce     = ['16–34', '35–54', '55–74', '75+']
risultati = {}

print("\n=== OLS — per fascia d'età ===")
for fascia in fasce:
    sub = df[df['fascia_eta'] == fascia].copy()
    mod = smf.ols(formula_full, data=sub).fit(cov_type='HC3')
    risultati[fascia] = mod

    b1  = mod.params['intensita_sm']
    b2  = mod.params['intensita_sm2']
    p1  = mod.pvalues['intensita_sm']
    p2  = mod.pvalues['intensita_sm2']
    r2  = mod.rsquared

    sig1 = '***' if p1<0.001 else ('**' if p1<0.01 else ('*' if p1<0.05 else 'n.s.'))
    sig2 = '***' if p2<0.001 else ('**' if p2<0.01 else ('*' if p2<0.05 else 'n.s.'))

    print(f"\n  Fascia {fascia}  (N={int(mod.nobs)}, R²={r2:.4f})")
    print(f"    intensita_sm   β={b1:.4f}  p={p1:.4f} {sig1}")
    print(f"    intensita_sm²  β={b2:.4f}  p={p2:.4f} {sig2}")

# ------------------------------------------------------------------
# 3.4  TABELLA COMPARATIVA β PER FASCIA
# ------------------------------------------------------------------
righe = []
for fascia in fasce:
    mod = risultati[fascia]
    for var in ['intensita_sm', 'intensita_sm2']:
        righe.append({
            'fascia'  : fascia,
            'variabile': var,
            'beta'    : round(mod.params[var], 4),
            'SE'      : round(mod.bse[var], 4),
            'p_value' : round(mod.pvalues[var], 4),
            'CI_low'  : round(mod.conf_int().loc[var, 0], 4),
            'CI_high' : round(mod.conf_int().loc[var, 1], 4),
            'sig'     : ('***' if mod.pvalues[var]<0.001
                         else ('**' if mod.pvalues[var]<0.01
                         else ('*' if mod.pvalues[var]<0.05 else 'n.s.')))
        })

tab = pd.DataFrame(righe)
print("\n=== Tabella comparativa β × fascia d'età ===")
print(tab.to_string(index=False))
tab.to_csv('output/dataset/step3_coefficienti.csv', index=False)
print("\nSalvato: output/dataset/step3_coefficienti.csv")

# ------------------------------------------------------------------
# 3.5  VERIFICA H₀ / H₁
# ------------------------------------------------------------------
print("\n=== Verifica ipotesi ===")
print("\nH₀: intensita_sm non associata a score_UCLA in nessuna fascia")
print("H₁: associazione presente e variabile per fascia d'età\n")

for fascia in fasce:
    mod  = risultati[fascia]
    p1   = mod.pvalues['intensita_sm']
    p2   = mod.pvalues['intensita_sm2']
    sig  = (p1 < 0.05) or (p2 < 0.05)
    esito = "→ associazione PRESENTE (contro H₀)" if sig else "→ associazione ASSENTE (a favore H₀)"
    print(f"  {fascia:6s}  intensita_sm p={p1:.4f} | intensita_sm² p={p2:.4f}  {esito}")

# ------------------------------------------------------------------
# 3.6  GRAFICI
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Fig A: β₁ e β₂ per fascia con intervalli di confidenza ---
x      = np.arange(len(fasce))
width  = 0.35
colori = ['#4C72B0', '#DD8452']

for i, var in enumerate(['intensita_sm', 'intensita_sm2']):
    betas  = [risultati[f].params[var] for f in fasce]
    ci_lo  = [risultati[f].conf_int().loc[var, 0] for f in fasce]
    ci_hi  = [risultati[f].conf_int().loc[var, 1] for f in fasce]
    yerr   = [[b - lo for b, lo in zip(betas, ci_lo)],
               [hi - b for b, hi in zip(betas, ci_hi)]]
    label  = 'β₁ (intensita_sm)' if i == 0 else 'β₂ (intensita_sm²)'
    axes[0].bar(x + i*width, betas, width, yerr=yerr, capsize=4,
                color=colori[i], alpha=0.8, label=label)

axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_xticks(x + width/2)
axes[0].set_xticklabels(fasce)
axes[0].set_title('Coefficienti OLS β₁ e β₂ per fascia d\'età')
axes[0].set_xlabel('Fascia d\'età')
axes[0].set_ylabel('Valore β (con IC 95%)')
axes[0].legend()

# --- Fig B: curve fitted per fascia ---
sm_range = np.linspace(0, 8, 100)
for fascia in fasce:
    mod = risultati[fascia]
    b0  = mod.params['Intercept']
    b1  = mod.params['intensita_sm']
    b2  = mod.params['intensita_sm2']
    # media dei controlli per fascia
    sub = df[df['fascia_eta'] == fascia]
    adj = (mod.params.get('sesso', 0)     * sub['sesso'].mean() +
           mod.params.get('education', 0) * sub['education'].mean() +
           mod.params.get('income', 0)    * sub['income'].mean())
    y_pred = b0 + b1*sm_range + b2*sm_range**2 + adj
    axes[1].plot(sm_range, y_pred, linewidth=2, label=fascia)

axes[1].axhline(df['score_UCLA'].mean(), color='gray',
                linestyle='--', linewidth=0.8)
axes[1].set_title('Curve OLS fitted per fascia d\'età')
axes[1].set_xlabel('Livello intensita_sm')
axes[1].set_ylabel('Score UCLA predetto')
axes[1].set_xticks(range(9))
axes[1].legend(title='Fascia d\'età')

plt.tight_layout()
fig.savefig('output/figures/step3_ols.png', dpi=150, bbox_inches='tight')
plt.show()
print("Salvato: output/figures/step3_ols.png")